# Assignment #1 -- Question 1: Chest X-ray CNN Classification
Colab runner. Code lives in the GitHub repo and is cloned below; this notebook only orchestrates it. All persistent state (dataset, manifests, checkpoints, results, figures) lives on Google Drive under `PROJECT_DIR` so it survives runtime disconnects -- re-running any cell resumes rather than restarts.

## 0. Setup: mount Drive, clone the repo, install deps

In [ ]:
# No Drive mount -- using local Colab storage instead. This means nothing here
# survives a runtime disconnect/restart; download report_notes.md, figures/, and
# the result CSVs via the download cell at the end of this notebook before you stop.
import os
PROJECT_DIR = '/content/generative_ai_assignment1'
os.makedirs(PROJECT_DIR, exist_ok=True)

from pathlib import Path
NOTES_PATH = Path(PROJECT_DIR) / 'report_notes.md'


In [ ]:
REPO_DIR = '/content/repo'
REPO_URL = 'https://github.com/i222502-school/generative_ai_assignment1.git'

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt

## 1. Kaggle dataset download
Get a Kaggle API token from https://www.kaggle.com/settings -> API -> Create New Token (the newer single-token format, starts with `KGAT_`).

In [ ]:
from getpass import getpass
import os

os.environ['KAGGLE_API_TOKEN'] = getpass('Kaggle API token (from kaggle.com/settings -> API): ')

import kagglehub
dataset_path = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
print(dataset_path)
!find {dataset_path} -maxdepth 2


## 2. Preprocessing + leakage-safe split (Tasks 1-2)
Manifests are cached to Drive -- dedup/patient-grouped split only runs once.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from pathlib import Path
import pandas as pd
from q1_cnn_xray.src.data import build_manifests, class_counts
from q1_cnn_xray.src.config import SplitRatios

SEED = 42
MANIFEST_DIR = Path(PROJECT_DIR) / 'manifests'

if (MANIFEST_DIR / 'train.csv').exists():
    splits = {name: pd.read_csv(MANIFEST_DIR / f'{name}.csv') for name in ('train', 'val', 'test')}
else:
    splits = build_manifests(Path(dataset_path), manifest_dir=MANIFEST_DIR, ratios=SplitRatios(), seed=SEED)

class_counts(splits)

from q1_cnn_xray.src.report_notes import append_section
append_section(NOTES_PATH, 'Dataset and split', class_counts(splits).to_markdown())

In [ ]:
import json
from q1_cnn_xray.src.stats import grayscale_mean_std

stats_path = Path(PROJECT_DIR) / 'grayscale_stats.json'
if stats_path.exists():
    stats = json.loads(stats_path.read_text())
    mean, std = stats['mean'], stats['std']
else:
    mean, std = grayscale_mean_std(splits['train']['path'])
    stats_path.write_text(json.dumps({'mean': mean, 'std': std}))
print(f'grayscale mean={mean:.4f} std={std:.4f}')

## 3. Primary CNN: architecture + layer table (Task 4)

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

from q1_cnn_xray.src.models.custom_cnn import PneumoniaCNN
from q1_cnn_xray.src.models.summary import layer_table, param_counts

primary_model = PneumoniaCNN()
table = layer_table(primary_model, (1, 224, 224))
table.to_csv(Path(PROJECT_DIR) / 'custom_cnn_layer_table.csv', index=False)
print(param_counts(primary_model))
table

append_section(
    NOTES_PATH, 'Primary CNN architecture',
    f'Parameter counts: {param_counts(primary_model)}\n\n' + table.to_markdown(index=False),
)

## 4. Primary CNN: default-hyperparameter training run (Task 6)
A sanity baseline before the full grid search below.

In [ ]:
from torch.utils.data import DataLoader
from q1_cnn_xray.src.dataset import ChestXrayDataset
from q1_cnn_xray.src.transforms import custom_cnn_transform, pretrained_transform
from q1_cnn_xray.src.class_balance import class_weight_tensor
from q1_cnn_xray.src.training.engine import fit

train_loader = DataLoader(
    ChestXrayDataset(splits['train'], custom_cnn_transform(train=True, mean=mean, std=std)),
    batch_size=32, shuffle=True,
)
val_loader = DataLoader(
    ChestXrayDataset(splits['val'], custom_cnn_transform(train=False, mean=mean, std=std)), batch_size=32
)

primary_model = PneumoniaCNN().to(device)
primary_result = fit(
    primary_model, train_loader, val_loader, lr=1e-3, max_epochs=30, patience=5, device=device,
    class_weights=class_weight_tensor(splits['train']),
    checkpoint_path=Path(PROJECT_DIR) / 'checkpoints' / 'primary_cnn_default.pth',
)
primary_result.history.plot(x='epoch', y=['train_loss', 'val_loss'], title='Primary CNN convergence')

append_section(
    NOTES_PATH, 'Primary CNN default-hyperparameter training',
    f'Epochs trained: {primary_result.epochs_trained}, best val loss: {primary_result.best_val_loss:.4f}\n\n'
    + primary_result.history.to_markdown(index=False),
)

## 5. Comparison baselines: VGG16 / ResNet50, frozen + fine-tuned (Task 5)

In [ ]:
from q1_cnn_xray.src.models.baselines import build_vgg16, build_resnet50, TrainMode

pretrained_train_loader = DataLoader(
    ChestXrayDataset(splits['train'], pretrained_transform(train=True)), batch_size=32, shuffle=True
)
pretrained_val_loader = DataLoader(
    ChestXrayDataset(splits['val'], pretrained_transform(train=False)), batch_size=32
)

baseline_specs = [
    ('vgg16_frozen', build_vgg16, TrainMode.FROZEN),
    ('vgg16_finetuned', build_vgg16, TrainMode.FINE_TUNED),
    ('resnet50_frozen', build_resnet50, TrainMode.FROZEN),
    ('resnet50_finetuned', build_resnet50, TrainMode.FINE_TUNED),
]

baseline_models = {}
for name, builder, mode in baseline_specs:
    baseline_model = builder(mode)
    ckpt = Path(PROJECT_DIR) / 'checkpoints' / f'{name}.pth'
    result = fit(
        baseline_model, pretrained_train_loader, pretrained_val_loader, lr=1e-4, max_epochs=20, patience=5,
        device=device, class_weights=class_weight_tensor(splits['train']), checkpoint_path=ckpt,
    )
    baseline_models[name] = baseline_model
    print(name, 'best_val_loss=', result.best_val_loss, 'epochs=', result.epochs_trained)

baseline_summary = pd.DataFrame(
    [
        {'model': name, **param_counts(baseline_models[name])}
        for name in baseline_models
    ]
)
append_section(NOTES_PATH, 'Comparison baselines (VGG16 / ResNet50)', baseline_summary.to_markdown(index=False))

## 6. Hyperparameter grid search (Task 7, part 1)
256 combos, resumable -- safe to interrupt and re-run this cell across sessions; already-completed combos (by hash) are skipped.

In [ ]:
from q1_cnn_xray.src.grid_search import run_grid_search, best_per_hyperparameter, full_grid

grid_results_path = Path(PROJECT_DIR) / 'grid_search_results.csv'
print(f'{len(full_grid())} total combos')
grid_results = run_grid_search(splits, mean, std, grid_results_path, device)
print(f'{len(grid_results)} combos complete')
best_per_hyperparameter(grid_results)

append_section(
    NOTES_PATH, 'Hyperparameter grid search',
    f'{len(grid_results)}/{len(full_grid())} combinations complete\n\n'
    + best_per_hyperparameter(grid_results).to_markdown(index=False),
)

## 7. Data-volume ablation (Task 7, part 2)

In [ ]:
from q1_cnn_xray.src.data_volume_study import run_data_volume_study

volume_results = run_data_volume_study(splits, mean, std, device)
volume_results.to_csv(Path(PROJECT_DIR) / 'data_volume_results.csv', index=False)
volume_results.plot(x='train_size', y='macro_f1', marker='o', title='Performance vs. training data volume')

append_section(NOTES_PATH, 'Data-volume ablation', volume_results.to_markdown(index=False))

## 8. Evaluation across all models (Task 8)

In [ ]:
from q1_cnn_xray.src.evaluate import evaluate_model, comparison_table

test_loader_custom = DataLoader(
    ChestXrayDataset(splits['test'], custom_cnn_transform(train=False, mean=mean, std=std)), batch_size=32
)
test_loader_pretrained = DataLoader(
    ChestXrayDataset(splits['test'], pretrained_transform(train=False)), batch_size=32
)

reports = [evaluate_model(primary_model, test_loader_custom, device, 'custom_cnn')]
reports += [
    evaluate_model(baseline_models[name], test_loader_pretrained, device, name) for name in baseline_models
]

comparison = comparison_table(*reports)
comparison.to_csv(Path(PROJECT_DIR) / 'model_comparison.csv')
comparison

append_section(NOTES_PATH, 'Model comparison (test set)', comparison.to_markdown())

## 9. Error analysis (Task 9)

In [ ]:
from q1_cnn_xray.src.error_analysis import top_misclassified, save_misclassified_grid

test_dataset_custom = ChestXrayDataset(splits['test'], custom_cnn_transform(train=False, mean=mean, std=std))
misclassified = top_misclassified(primary_model, test_dataset_custom, device, n=10)
save_misclassified_grid(misclassified, Path(PROJECT_DIR) / 'figures' / 'misclassified.png')
misclassified

append_section(
    NOTES_PATH, 'Error analysis (top misclassified test cases)',
    misclassified.to_markdown(index=False) + '\n\nGrid image: figures/misclassified.png',
)

## Download results
Run this last -- zips everything except checkpoints and downloads it.

In [ ]:
import shutil
from google.colab import files

# Excludes checkpoints/ -- grid search alone can produce hundreds of small files
# there, and none of it is needed to write the report.
export_dir = Path('/content/report_export')
if export_dir.exists():
    shutil.rmtree(export_dir)
shutil.copytree(PROJECT_DIR, export_dir, ignore=shutil.ignore_patterns('checkpoints'))

archive_path = shutil.make_archive('/content/report_export', 'zip', export_dir)
files.download(archive_path)
